In [2]:
import json
import uuid
import csv
import pandas as pd

In [18]:
input_file = "../Data/test_KTH_only.json"
authors_with_pids = "data/authors_with_pids.csv"
keywords_with_pids="data/keywords_with_pids.csv"
author_graph_file="../coauthorsgraph/graph/graph.csv"
id_to_author_file ="../coauthorsgraph/graph/id-to-author.csv"

## Create Dataframes

In [47]:
author_ids_df = pd.read_csv(id_to_author_file, sep=';')
authors_info_df = pd.read_csv(authors_with_pids, sep=';')
keywords_pids_df = pd.read_csv(keywords_with_pids, sep=';')
author_graph_df = pd.read_csv(author_graph_file, sep=';')

In [53]:
print(author_graph_df.dtypes)
author_graph_df.head()

node                 int64
1_hop_neighbours    object
2_hop_neighbours    object
3_hop_neighbours    object
dtype: object


,node,1_hop_neighbours,2_hop_neighbours,3_hop_neighbours
0,0,[1],[],[]
1,1,[0],[],[]
2,2,[3],[],[]
3,3,[2],[],[]
4,4,[5],[],[]


In [20]:
# Merge the dataframes on the 'id' column from author_ids_df and 'ID' column from authors_info_df
# Convert the 'id' column in author_ids_df to string to match the 'ID' column in authors_info_df
author_ids_df['id'] = author_ids_df['id'].astype(str)

# Merge the dataframes on the 'id' column from author_ids_df and 'ID' column from authors_info_df
merged_authors_df = pd.merge(author_ids_df, authors_info_df, left_on='id', right_on='ID', how='inner')

# Display the resulting dataframe
merged_authors_df.head()

,id,name,Author,ID,PIDs,Keywords
0,0,"STRÖMBERG, PHILIP","Strömberg, Philip",0,1214481,NaN
1,1,"BLOMKVIST KARLSSON, VERA","Blomkvist Karlsson, Vera",1,"1214481,1608961","Clang,Code generation,CUDA,Debugging,Parallel ..."
2,2,"RENMAN, CASPER","Renman, Casper",2,810264,NaN
3,3,"FRISTEDT, HAMPUS","Fristedt, Hampus",3,"810264,1113057","Deep learning,Convolutional neural network,hom..."
4,4,"ISHII, SHOTARO","Ishii, Shotaro",4,1597519,NaN


In [29]:
keywords_pids_df["PID"] = keywords_pids_df["PID"].astype(str)
keywords_pids_df.dtypes

Keyword    object
PID        object
dtype: object

We will be treating each keyword as a query from each author. Each row will have query_id, author_id, query, PID, and score (0-3) ## TO DO SCORE

In [56]:
query_id = 0
rows =  []
for idx, author in merged_authors_df.iterrows():
    if pd.isna(author["Keywords"]):  # Check if Keywords is NaN, ignore
        continue
    splitKeywords = author["Keywords"].split(',')
    for keyword in splitKeywords:
        keyword_row = keywords_pids_df[keywords_pids_df['Keyword'] == keyword]
        keyword_pids = keyword_row["PID"]
        count = 0 
        for idx, pids in keyword_pids.items():
            split_pids = pids.split(',')
            for pid in split_pids:
                rows.append(
                    (query_id, author["id"], keyword, pid)
                )
        query_id+=1

expanded_df = pd.DataFrame(rows, columns=["query_id", "author_id", "query", "PID"])
expanded_df.head()

,query_id,author_id,query,PID
0,0,1,Clang,1109257
1,0,1,Clang,1608961
2,0,1,Clang,1608961
3,1,1,Code generation,1892787
4,1,1,Code generation,1608961


# Adding scores

In [67]:
def compute_score(row):
    # Check if the row["PID"] was written by the author
    if row["PID"] in str(author["PIDs"]).split(','):
        return 3

    author_row = merged_authors_df[merged_authors_df['PIDs'].str.contains(row["PID"], na=False)]
    if not author_row.empty:
        author_id = author_row.iloc[0]['id']
    else:
        author_id = None
    if author_id == None or row["author_id"] == None:
        return 0
    # if author_id found in 1_hop_neighbors -> 3
    # Find the row in author_graph_df where the node matches author_id
    author_graph_row = author_graph_df[author_graph_df["node"] == int(author_id)]
    if not author_graph_row.empty:
        if row["author_id"] in author_graph_row.iloc[0]["1_hop_neighbours"]:
            return 3
        elif row["author_id"] in author_graph_row.iloc[0]["2_hop_neighbours"]:
            return 2
        elif row["author_id"] in author_graph_row.iloc[0]["3_hop_neighbours"]:
            return 1
        return 3
    # elif row["author_id"] in author_graph_df.loc[author_id, "2_hop_neighbours"]:
    #     return 2
    # elif row["author_id"] in author_graph_df.loc[author_id, "3_hop_neighbours"]:
    #     return 1
    # if 2 hop -> 2
    # if 3 hop -> 1
    # else 0
    return 0

expanded_df["score"] = expanded_df.apply(compute_score, axis=1)

expanded_df.head()

,query_id,author_id,query,PID,score
0,0,1,Clang,1109257,0
1,0,1,Clang,1608961,3
2,0,1,Clang,1608961,3
3,1,1,Code generation,1892787,0
4,1,1,Code generation,1608961,3


In [68]:
expanded_df.to_csv("data/query_data.csv", index=False)